# Setup and Library Verification

This notebook installs and verifies all Python libraries needed for the re-analysis of
[Semantically Incongruent or Congruent Eggplants](https://pmc.ncbi.nlm.nih.gov/articles/PMC9540983/).

Analyses covered:
1. **Lexical frequency** – how common each word is in the language
2. **Phonology** – phoneme-level properties of stimuli
3. **Next-word prediction / surprisal** – language-model-based predictability
4. **Sentence-level semantic metrics** – semantic similarity / relatedness between sentences

Run the *Installation* section once (or inside a virtual environment), then re-run
the *Import & Verification* sections at the start of every analysis session.

---
## 1  Installation

Uncomment the cell below and run it once to install all required packages.
All pinned versions reflect the latest stable releases as of April 2026.

In [ ]:
# Uncomment and run once to install dependencies
# import sys
#
# # --- core scientific stack ---
# !{sys.executable} -m pip install numpy pandas matplotlib seaborn scipy
#
# # --- lexical frequency ---
# !{sys.executable} -m pip install wordfreq nltk
#
# # --- phonology ---
# !{sys.executable} -m pip install pronouncing eng-to-ipa phonemizer
#
# # --- next-word prediction / surprisal (HuggingFace + PyTorch) ---
# !{sys.executable} -m pip install transformers torch
#
# # --- sentence-level semantic metrics ---
# !{sys.executable} -m pip install sentence-transformers spacy gensim
# !{sys.executable} -m spacy download en_core_web_sm

---
## 2  Basic Scientific Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats

# Notebook display settings
%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print(f"numpy   {np.__version__}")
print(f"pandas  {pd.__version__}")
print(f"scipy   {scipy.__version__}")
print(f"seaborn {sns.__version__}")
print("Basic scientific libraries OK")

---
## 3  Lexical Frequency

| Library | Purpose |
|---|---|
| [`wordfreq`](https://github.com/rspeer/wordfreq) | Word frequency norms for 40+ languages (Zipf scale) |
| [`nltk`](https://www.nltk.org/) | General NLP toolkit; `FreqDist`, corpus access |

In [ ]:
# wordfreq ----------------------------------------------------------------
from wordfreq import word_frequency, zipf_frequency

word = "eggplant"
freq   = word_frequency(word, "en")
zipf   = zipf_frequency(word, "en")
print(f"'{word}'  raw freq = {freq:.2e}  |  Zipf = {zipf:.2f}")

# nltk --------------------------------------------------------------------
import nltk
nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)
nltk.download("stopwords",  quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
from nltk.probability import FreqDist

print(f"nltk    {nltk.__version__}")
print("Lexical frequency libraries OK")

---
## 4  Phonology

| Library | Purpose |
|---|---|
| [`pronouncing`](https://pronouncing.readthedocs.io/) | CMU Pronouncing Dictionary (ARPAbet phones, syllable count, stress) |
| [`eng_to_ipa`](https://github.com/mphilli/English-to-IPA) | English grapheme → IPA transcription |
| [`phonemizer`](https://github.com/bootphon/phonemizer) | Broad phonemization (eSpeak backend); multi-language |

In [ ]:
# pronouncing -------------------------------------------------------------
import pronouncing

phones = pronouncing.phones_for_word("eggplant")
print(f"ARPAbet phones  : {phones}")
if phones:
    print(f"Syllable count  : {pronouncing.syllable_count(phones[0])}")
    print(f"Stresses        : {pronouncing.stresses(phones[0])}")

# eng_to_ipa --------------------------------------------------------------
import eng_to_ipa as ipa

ipa_str = ipa.convert("eggplant")
print(f"IPA             : {ipa_str}")

# phonemizer --------------------------------------------------------------
# Requires eSpeak-NG system library (apt install espeak-ng)
try:
    from phonemizer import phonemize
    result = phonemize("eggplant", backend="espeak", language="en-us", with_stress=True)
    print(f"phonemizer IPA  : {result.strip()}")
except Exception as exc:
    print(f"phonemizer      : skipped ({exc})")

print("Phonology libraries OK")

---
## 5  Next-Word Prediction / Surprisal

| Library | Purpose |
|---|---|
| [`transformers`](https://huggingface.co/docs/transformers/) | HuggingFace model hub; GPT-2 / GPT-2-large for surprisal |
| [`torch`](https://pytorch.org/) | PyTorch backend |

Surprisal (–log₂ P(w | context)) is the standard measure of
next-word predictability used in psycholinguistics / ERP research.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import math

print(f"transformers {transformers.__version__}")
print(f"torch        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Load GPT-2 (downloads ~500 MB on first run)
_LM_NAME = "gpt2"
_lm_tokenizer = AutoTokenizer.from_pretrained(_LM_NAME)
_lm_model     = AutoModelForCausalLM.from_pretrained(_LM_NAME)
_lm_model.eval()

def word_surprisal(context: str, target: str, model=_lm_model,
                   tokenizer=_lm_tokenizer) -> float:
    """Return surprisal (bits) of *target* given *context*.

    Parameters
    ----------
    context : str
        The sentence fragment preceding the target word (no trailing space needed).
    target : str
        The word whose surprisal is to be computed.

    Returns
    -------
    float
        Surprisal in bits ( –log2 P(target | context) ).
    """
    full_text  = context + " " + target
    enc_full   = tokenizer(full_text,   return_tensors="pt")
    enc_ctx    = tokenizer(context + " ", return_tensors="pt")
    n_ctx_toks = enc_ctx["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model(**enc_full, labels=enc_full["input_ids"])
        logits  = outputs.logits  # (1, seq_len, vocab)

    log_probs = torch.nn.functional.log_softmax(logits[0], dim=-1)
    target_ids = enc_full["input_ids"][0, n_ctx_toks:]
    total_log2 = sum(
        log_probs[n_ctx_toks - 1 + i, tok_id].item() / math.log(2)
        for i, tok_id in enumerate(target_ids)
    )
    return -total_log2


# Quick demo
ctx_congruent   = "She put the keys in her"
ctx_incongruent = "She wrote the code in her"
target = "eggplant"

s_cong   = word_surprisal(ctx_congruent,   target)
s_incong = word_surprisal(ctx_incongruent, target)
print(f"Surprisal of '{target}' after congruent context   : {s_cong:.2f} bits")
print(f"Surprisal of '{target}' after incongruent context : {s_incong:.2f} bits")
print("Next-word prediction libraries OK")

---
## 6  Sentence-Level Semantic Metrics

| Library | Purpose |
|---|---|
| [`sentence-transformers`](https://www.sbert.net/) | Sentence embeddings (SBERT); cosine similarity |
| [`spacy`](https://spacy.io/) | NLP pipeline; token vectors, dependency parse |
| [`gensim`](https://radimrehurek.com/gensim/) | Word2Vec / FastText vectors; WMD distance |

In [ ]:
# sentence-transformers ---------------------------------------------------
from sentence_transformers import SentenceTransformer, util

# Lightweight model; downloads ~90 MB on first run
_sbert = SentenceTransformer("all-MiniLM-L6-v2")

def sentence_cosine(s1: str, s2: str, model=_sbert) -> float:
    """Cosine similarity (–1 … 1) between two sentences."""
    emb = model.encode([s1, s2], convert_to_tensor=True)
    return float(util.cos_sim(emb[0], emb[1]))

sent_a = "The girl ate an eggplant for dinner."
sent_b = "The boy had vegetables in the evening."
sent_c = "The rocket launched into outer space."

print(f"cos(a, b) = {sentence_cosine(sent_a, sent_b):.3f}  (semantically related)")
print(f"cos(a, c) = {sentence_cosine(sent_a, sent_c):.3f}  (semantically unrelated)")
print("sentence-transformers OK")

In [ ]:
# spaCy -------------------------------------------------------------------
import spacy

# Load small English model (install: python -m spacy download en_core_web_sm)
nlp = spacy.load("en_core_web_sm")

doc = nlp("She put the eggplant on the table.")
print("Token | POS | Dep | IsStop")
for tok in doc:
    print(f"  {tok.text:<12} {tok.pos_:<6} {tok.dep_:<10} {tok.is_stop}")
print(f"spaCy version: {spacy.__version__}")
print("spaCy OK")

In [ ]:
# gensim ------------------------------------------------------------------
import gensim
from gensim.downloader import load as gensim_load

print(f"gensim {gensim.__version__}")

# Load a small pre-trained word-vector model (~66 MB) for quick demos.
# Replace with 'word2vec-google-news-300' for full-size vectors.
print("Loading 'glove-wiki-gigaword-50' vectors (~66 MB, first run only) …")
_wv = gensim_load("glove-wiki-gigaword-50")

pairs = [("eggplant", "vegetable"), ("eggplant", "rocket")]
for w1, w2 in pairs:
    sim = _wv.similarity(w1, w2) if w1 in _wv and w2 in _wv else float("nan")
    print(f"  similarity('{w1}', '{w2}') = {sim:.3f}")

print("gensim OK")

---
## 7  Summary

Run this cell as a quick all-pass check before starting any analysis.

In [ ]:
import importlib, sys

required = [
    # (import_name, display_name)
    ("numpy",               "numpy"),
    ("pandas",              "pandas"),
    ("matplotlib",          "matplotlib"),
    ("seaborn",             "seaborn"),
    ("scipy",               "scipy"),
    ("wordfreq",            "wordfreq"),
    ("nltk",                "nltk"),
    ("pronouncing",         "pronouncing"),
    ("eng_to_ipa",          "eng_to_ipa"),
    ("phonemizer",          "phonemizer"),
    ("transformers",        "transformers"),
    ("torch",               "torch"),
    ("sentence_transformers","sentence-transformers"),
    ("spacy",               "spaCy"),
    ("gensim",              "gensim"),
]

all_ok = True
for mod, label in required:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "n/a")
        print(f"  ✅  {label:<24} {ver}")
    except ImportError as e:
        print(f"  ❌  {label:<24} NOT FOUND ({e})")
        all_ok = False

print()
print("All libraries available ✅" if all_ok else "Some libraries are missing ❌  – see Section 1 for install instructions.")